## Environment setup

In [ ]:
# Clean reset
!pip uninstall -y transformers peft accelerate -q

# Install compatible versions
!pip install -q \
    transformers==4.41.2 \
    accelerate==0.30.1 \
    peft==0.11.1 \
    sentencepiece

In [2]:
!pip install -q seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Verify versions

In [3]:
import transformers, peft, accelerate
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.41.2
peft: 0.11.1
accelerate: 0.30.1


## Copy project to writable working dir

In [5]:
import os, shutil

SRC  = "/kaggle/input/datasets/ajaykumarprasad123/jd-bias-detector-cleaned"
DEST = "/kaggle/working/jd-bias-detector"

if os.path.exists(DEST):
    shutil.rmtree(DEST)

shutil.copytree(SRC, DEST)
os.chdir(DEST)

print("CWD:", os.getcwd())
print("Files:", os.listdir("."))

CWD: /kaggle/working/jd-bias-detector
Files: ['data', 'training']


## Optional sanity check on dataset

In [6]:
import json, random
from collections import Counter, defaultdict

DATA_DIR = "data/annotated"

def load_split(split):
    path = f"{DATA_DIR}/{split}.jsonl"
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train = load_split("train")
val   = load_split("val")
test  = load_split("test")

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

sample = random.choice(train)
print("\nSample text:", sample["text"][:200])
print("Sample labels:", sample["labels"][:20])

def bias_ratio(data):
    biased = sum(1 for s in data if any(l != "O" for l in s["labels"]))
    return biased, len(data) - biased

b, n = bias_ratio(train)
print(f"\nBias vs Neutral (Train): {b} biased | {n} neutral")

Train: 2554 | Val: 309 | Test: 315

Sample text: We are hiring a Product Manager to join a remote-first organization. You will collaborate with cross-functional partners to deliver customer value. You will communicate clearly with stakeholders and t
Sample labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Bias vs Neutral (Train): 1307 biased | 1247 neutral


## Disable WandB in config if needed

In [7]:
from pathlib import Path

cfg_path = Path("training/configs/deberta_base.yaml")
text = cfg_path.read_text()

if "report_to:" not in text:
    text += "\nreport_to: none\n"
else:
    import re
    text = re.sub(r"report_to\s*:\s*.*", "report_to: none", text)

cfg_path.write_text(text)
print(cfg_path.read_text())

# ─────────────────────────────────────────────────────────────
#  DeBERTa-v3-base  |  JD Bias Token Classifier
#  Optimised for cloud GPU (T4 / V100 / A100)
# ─────────────────────────────────────────────────────────────

# ── Model ─────────────────────────────────────────────────────
base_model:  microsoft/deberta-v3-base
output_dir:  models/deberta-jd-bias-v1

# ── Training schedule ─────────────────────────────────────────
epochs:                 5
batch_size:             16          # T4 (16GB): 16 | A100: 32
grad_accum:             2           # effective batch = batch_size × grad_accum
lr:                     2.0e-5
weight_decay:           0.01
warmup_ratio:           0.1
lr_scheduler:           linear
max_grad_norm:          1.0

# ── Sequence length ───────────────────────────────────────────
max_length:             256         # Most JDs < 256 tokens; saves memory vs 512

# ── Precision (set bf16: true for A100 / H100) ───────────────
fp16:                   true
bf16:      

## Train on the current cleaned dataset

In [8]:
!python -m training.train --config training/configs/deberta_base.yaml

2026-03-20 11:22:08.097065: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774005728.320535     147 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774005728.386779     147 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774005728.918377     147 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774005728.918450     147 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774005728.918457     147 computation_placer.cc:177] computation placer alr

## Remove checkpoints after training

In [9]:
import os, shutil

base = "models/deberta-jd-bias-v1"

if os.path.exists(base):
    for f in os.listdir(base):
        path = os.path.join(base, f)
        if f.startswith("checkpoint") and os.path.isdir(path):
            shutil.rmtree(path)

print("Remaining files:", os.listdir(base))

Remaining files: ['config.json', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'model.safetensors', 'special_tokens_map.json', 'added_tokens.json', 'spm.model']


## Evaluate and regenerate report

In [10]:
!python -m training.evaluate \
  --model_dir models/deberta-jd-bias-v1 \
  --test_data data/annotated/test.jsonl \
  --output docs/evaluation_report.md

2026-03-20 11:30:03.157568: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774006203.183103    1348 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774006203.191156    1348 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774006203.210414    1348 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774006203.210456    1348 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774006203.210462    1348 computation_placer.cc:177] computation placer alr

## Test the newly trained model

In [11]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model="/kaggle/working/jd-bias-detector/models/deberta-jd-bias-v1",
    aggregation_strategy="simple",
)

def clean_word(w):
    return w.replace("##", "").replace("Ġ", "").strip()

test_jds = [
    "We need someone who thrives in a demanding environment and loves intense challenges.",
    "Looking for a young, hungry developer who can crush it at a fast-paced startup.",
    "The ideal candidate is a collaborative team player who communicates clearly and supports their colleagues.",
    "We want a rockstar engineer with ninja-level skills who eats, sleeps and breathes code.",
    "Join our diverse team as a senior engineer. You will mentor others and drive technical decisions.",
    "We need someone who can adapt quickly in a dynamic work setting.",
    "Looking for candidates early in their career with strong motivation.",
    "We want someone who can handle a high workload and changing priorities.",
]

for jd in test_jds:
    results = ner(jd)
    print(f"\nJD: {jd}")
    if results:
        for r in results:
            word = clean_word(r["word"])
            print(f"  [{r['entity_group']}] '{word}' ({r['score']:.2f})")
    else:
        print("  No bias detected")

2026-03-20 11:30:37.589059: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774006237.617501      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774006237.625859      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774006237.647075      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774006237.647131      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774006237.647134      55 computation_placer.cc:177] computation placer alr


JD: We need someone who thrives in a demanding environment and loves intense challenges.
  [ABILITY_CODED] 'demanding environment' (0.88)
  [ABILITY_CODED] 'loves intense challenges' (0.69)

JD: Looking for a young, hungry developer who can crush it at a fast-paced startup.
  [AGEIST] 'young, hungry' (0.83)
  [GENDER_CODED] 'crush it' (0.80)
  [ABILITY_CODED] 'fast' (0.83)
  [ABILITY_CODED] 'paced' (0.93)

JD: The ideal candidate is a collaborative team player who communicates clearly and supports their colleagues.
  No bias detected

JD: We want a rockstar engineer with ninja-level skills who eats, sleeps and breathes code.
  [EXCLUSIONARY] 'rockstar' (0.64)
  [EXCLUSIONARY] 'ninja' (0.53)

JD: Join our diverse team as a senior engineer. You will mentor others and drive technical decisions.
  No bias detected

JD: We need someone who can adapt quickly in a dynamic work setting.
  [ABILITY_CODED] 'quickly' (0.36)
  [ABILITY_CODED] 'dynamic work setting' (0.84)

JD: Looking for candida

## Create a lightweight downloadable model folder

In [12]:
import os, shutil

SRC = "models/deberta-jd-bias-v1"
DEST = "models/deberta-jd-bias-v1-clean"

if os.path.exists(DEST):
    shutil.rmtree(DEST)
os.makedirs(DEST, exist_ok=True)

important_files = [
    "config.json",
    "model.safetensors",
    "pytorch_model.bin",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "spm.model",
    "added_tokens.json",
]

copied = []
for f in important_files:
    src_path = os.path.join(SRC, f)
    if os.path.exists(src_path):
        shutil.copy(src_path, DEST)
        copied.append(f)

print("Copied:", copied)
print("Final files:", os.listdir(DEST))

Copied: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json', 'spm.model', 'added_tokens.json']
Final files: ['config.json', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors', 'special_tokens_map.json', 'added_tokens.json', 'spm.model']


## Zip the clean model for download

In [13]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/deberta-jd-bias-v1-clean",
    "zip",
    root_dir="/kaggle/working/jd-bias-detector/models/deberta-jd-bias-v1-clean"
)

print("Created:", zip_path)

Created: /kaggle/working/deberta-jd-bias-v1-clean.zip
